# LLMDTA_FNet -- Component Ablation (chay thu / smoke test) -- Davis/Warm (Kaggle Notebook)

Notebook nay chay THU (trial run) script `code/ablation_fft.py` -- ablation tung thanh phan kien truc cua model FNet (model dung FourierMixing/FFT thay CrossAttention): Encoder, chinh co che FFT mixing, Self-Attention Pooling, Residual Fusion, MoE, va bien the doi mixing sang CrossAttention.

**8 bien the duoc ablation:**
- `full` -- Model FFT day du (baseline)
- `wo_encoder` -- Linear Encoder thay 1D-CNN (giu FFT)
- `wo_fft_mixing` -- Bo han FFT (Identity passthrough)
- `attn_mixing` -- FFT -> CrossAttention (so sanh mixing)
- `wo_self_attn_pool` -- Mean Pooling thay Self-Attention Pooling (giu FFT)
- `wo_residual` -- Bo residual fusion (giu FFT)
- `wo_moe` -- 1 predictor don thay MoE (giu FFT)
- `linear_only` -- Chi Linear predictor (khong Encoder/Mixing/MoE)

**Muc dich cua ban "chay thu" nay**: xac nhan script chay dung, khong loi, tren toan bo 8 bien the, VOI SO EPOCH RAT NHO (mac dinh 5 epoch/bien the, patience 3) de nhanh co ket qua so bo. Ket qua so bo nay KHONG dai dien cho hieu nang thuc su cua model (can it nhat epochs~100, patience~20, va chay ca 5 fold moi co the ket luan).

Sau khi chay thu thanh cong (khong loi, thay bang ket qua o cuoi notebook), tang `TRIAL_EPOCHS` / `TRIAL_PATIENCE`, bat `--all_folds`, de chay ablation day du (co the mat vai gio tuy GPU va so bien the).

## Truoc khi chay (bat buoc)

1. **Bat GPU**: panel ben phai -> *Settings* -> *Accelerator* -> chon **GPU T4 x2** (hoac P100).
2. **Bat Internet**: *Settings* -> *Internet* -> **On** (can de `pip install` va `git clone`).
3. **Add Data**: nut **+ Add Data** (goc phai) -> tim `llmdta` (chu so huu `christang0002`) -> **Add**.

Sau khi lam du 3 buoc tren, chon **Run All**.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Luu y: KHONG pin gensim==4.3.1 (khong co wheel dung san cho Python tren Kaggle
# -> pip phai build tu source va thuong loi). De pip tu chon ban gensim moi nhat.
!pip install -q rdkit gensim mol2vec wandb scikit-learn scipy tqdm
print('Da cai xong dependencies.')

In [ ]:
import os

REPO_URL = 'https://github.com/glucose20org/Temp.git'
REPO_BRANCH = 'quantum'
REPO_ROOT = '/kaggle/working/Temp'

if not os.path.exists(REPO_ROOT):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}
else:
    print('Repo da ton tai, dang pull ban moi nhat...')
    !cd {REPO_ROOT} && git pull origin {REPO_BRANCH}

os.chdir(REPO_ROOT)
print('Working dir:', os.getcwd())

assert os.path.exists('code/ablation_fft.py'), 'Khong tim thay code/ablation_fft.py -- kiem tra lai REPO_BRANCH.'
print('Da tim thay code/ablation_fft.py')

## Tim du lieu pretrain embedding tu dataset da Add Data

Cell duoi tu dong quet toan bo `/kaggle/input/**` de tim 2 file `*_drug_pretrain.pkl` va `*_esm_pretrain.pkl` cho dataset `davis`. Neu ten file trong dataset Kaggle khac pattern doan duoc, notebook se in ra cay thu muc `/kaggle/input` de ban tu xac dinh duong dan, roi gan thu cong vao `MANUAL_DRUG_PKL` / `MANUAL_PROT_PKL` o cell ke tiep.

In [ ]:
import glob, shutil, tarfile

DATASET = 'davis'         # davis | kiba | metz
RUNNING_SET = 'warm'      # warm | novel-drug | novel-prot | novel-pair

# Fold-data (train/valid/test csv) da co san trong repo duoi dang .tar.gz -> chi can giai nen
fold_root = os.path.join(REPO_ROOT, 'data', 'dta-5fold-dataset')
tar_path = os.path.join(fold_root, f'{DATASET}.tar.gz')
extracted_path = os.path.join(fold_root, DATASET)
if not os.path.exists(extracted_path) and os.path.exists(tar_path):
    print(f'Giai nen {tar_path} ...')
    with tarfile.open(tar_path) as tf:
        try:
            tf.extractall(fold_root, filter='data')
        except TypeError:
            tf.extractall(fold_root)  # Python < 3.12 khong co tham so filter
print('Fold-data san sang tai:', extracted_path)

# Ghi de thu cong neu can (de trong '' de dung auto-detect)
MANUAL_DRUG_PKL = ''
MANUAL_PROT_PKL = ''

KAGGLE_INPUT_ROOT = '/kaggle/input'

def find_best_match(patterns, search_root):
    candidates = []
    for pat in patterns:
        candidates += glob.glob(os.path.join(search_root, '**', pat), recursive=True)
    return sorted(set(candidates))

target_dir = os.path.join(REPO_ROOT, 'data', DATASET)
os.makedirs(target_dir, exist_ok=True)
target_drug = os.path.join(target_dir, f'{DATASET}_drug_pretrain.pkl')
target_prot = os.path.join(target_dir, f'{DATASET}_esm_pretrain.pkl')

if MANUAL_DRUG_PKL:
    shutil.copy(MANUAL_DRUG_PKL, target_drug)
    print('Da copy (thu cong) drug pretrain ->', target_drug)
elif not os.path.exists(target_drug):
    drug_candidates = find_best_match([f'*{DATASET}*drug_pretrain*.pkl', f'*{DATASET}*mol2vec*.pkl'], KAGGLE_INPUT_ROOT)
    print('Ung vien drug pretrain:', drug_candidates)
    if drug_candidates:
        shutil.copy(drug_candidates[0], target_drug)
        print('Da copy drug pretrain ->', target_drug)

if MANUAL_PROT_PKL:
    shutil.copy(MANUAL_PROT_PKL, target_prot)
    print('Da copy (thu cong) prot pretrain ->', target_prot)
elif not os.path.exists(target_prot):
    prot_candidates = find_best_match([f'*{DATASET}*esm_pretrain*.pkl', f'*{DATASET}*esm*.pkl'], KAGGLE_INPUT_ROOT)
    print('Ung vien prot pretrain:', prot_candidates)
    if prot_candidates:
        shutil.copy(prot_candidates[0], target_prot)
        print('Da copy prot pretrain ->', target_prot)

if not os.path.exists(target_drug) or not os.path.exists(target_prot):
    print('\nKhong tu dong tim thay du file pretrain. Cay thu muc /kaggle/input hien co:')
    for root, dirs, fs in os.walk(KAGGLE_INPUT_ROOT):
        depth = root.replace(KAGGLE_INPUT_ROOT, '').count(os.sep)
        if depth > 3:
            continue
        print('  ' * depth + os.path.basename(root) + '/')
        for f in fs[:15]:
            print('  ' * (depth + 1) + f)
    print('\nHay kiem tra duong dan chinh xac roi gan vao MANUAL_DRUG_PKL / MANUAL_PROT_PKL o tren va chay lai cell nay.')
    print('Neu chua Add Data: bam "+ Add Data" -> tim "llmdta" (christang0002/llmdta) -> Add.')
else:
    print('\nDa co du 2 file pretrain embedding cho', DATASET)

## Chay thu (trial run) toan bo 8 bien the ablation

Cell duoi goi truc tiep `code/ablation_fft.py` (script da duoc push len GitHub o branch `quantum`) voi so epoch RAT NHO de kiem tra nhanh: khong loi, moi bien the train/eval duoc, va co bang ket qua o cuoi.

- `TRIAL_EPOCHS = 5`, `TRIAL_PATIENCE = 3`: chi de smoke-test, khong phai con so dung de bao cao ket qua.
- Chay 1 fold (`FOLD = 0`) thay vi `--all_folds` de nhanh hon.
- Neu muon test nhanh hon nua, sua `--variants` de chi chay vai bien the (vi du `full,wo_fft_mixing,attn_mixing`).

In [ ]:
DATASET = 'davis'
RUNNING_SET = 'warm'
FOLD = 0

TRIAL_EPOCHS = 5      # CHI DE CHAY THU -- tang len ~100 khi chay that
TRIAL_PATIENCE = 3    # CHI DE CHAY THU -- tang len ~20 khi chay that
TRIAL_BATCH_SIZE = 64
OUTPUT_DIR = '/kaggle/working/ablation_trial_results'

os.makedirs(OUTPUT_DIR, exist_ok=True)

!python code/ablation_fft.py \
    --dataset {DATASET} --running_set {RUNNING_SET} --fold {FOLD} \
    --epochs {TRIAL_EPOCHS} --patience {TRIAL_PATIENCE} --batch_size {TRIAL_BATCH_SIZE} \
    --output_dir {OUTPUT_DIR}

## Doc ket qua chay thu

Neu cell tren chay xong khong bao loi, ket qua (CSV + JSON) da duoc luu vao `OUTPUT_DIR`. Cell duoi doc file CSV moi nhat va hien bang so sanh CI/MSE giua 8 bien the.

In [ ]:
import glob
import pandas as pd

csv_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, f'ablation_fnet_{DATASET}_{RUNNING_SET}_fold{FOLD}_*.csv')))
assert csv_files, 'Khong tim thay file ket qua -- kiem tra lai cell chay ablation o tren co bao loi khong.'
result_csv = csv_files[-1]
print('Doc ket qua tu:', result_csv)

df = pd.read_csv(result_csv)
df_sorted = df.sort_values('ci', ascending=False).reset_index(drop=True)
df_sorted[['variant', 'description', 'ci', 'mse', 'r2', 'best_epoch', 'training_time']]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.bar(df_sorted['variant'], df_sorted['ci'], color='#4C72B0')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Test CI')
plt.title(f'FFT Component Ablation (TRIAL, {TRIAL_EPOCHS} epoch/bien the) - {DATASET}-{RUNNING_SET} fold {FOLD}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'trial_ci_comparison.png'), dpi=150)
plt.show()

## Ket qua & buoc tiep theo

Day chi la **chay thu** (1 fold, 5 epoch/bien the) de xac nhan `ablation_fft.py` hoat dong dung tren Kaggle -- CI/MSE o day chi mang tinh minh hoa, chua the dung de ket luan thanh phan nao quan trong.

**De chay ablation THAT (dung de bao cao ket qua):**
1. Tang `TRIAL_EPOCHS` len ~100 va `TRIAL_PATIENCE` len ~20.
2. Them `--all_folds` vao lenh `!python code/ablation_fft.py ...` de chay ca 5 fold va lay mean +/- std.
3. Luu y thoi gian chay: 8 bien the x 5 fold x ~100 epoch se mat kha lau -- nen chay tren session GPU rieng (giong cach 3 notebook `Kaggle_*_Davis_Warm.ipynb` khac tach rieng tung model de tranh vuot quota).
4. Sau khi chay xong, vao tab **Output** cua Kaggle de tai `ablation_fnet_*.csv` / `*.json` / `*_AGGREGATED_*.csv` ve.